# 03 · Base models (rebuilding the previous best recipe)

Four model families that earned their place in the earlier rounds, retrained with clean code:

| Family | Features | Why it is here |
| --- | --- | --- |
| LightGBM | base (55) | fast, strong gradient-boosted trees; different tree growth from CatBoost |
| CatBoost | base | strongest single family; ordered boosting handles the categoricals well |
| CatBoost-recon | base + reconstructed scores (61) | adds "observed vs expected external score" information |
| TabM | base | neural network; weaker alone but makes *different* ranking errors, which helps the blend |

**Validation protocol.** 5-fold stratified CV on the 171,202 training rows; AUC on out-of-fold (OOF)
predictions. Test predictions are the average of the 5 fold models.
1. *Development run* (seed 0): the validation fold is used for early stopping, which gives a good number of
   trees but a slightly optimistic score.
2. *Final runs* (seeds 0, 1, 2): fixed training length = median best iteration of the development run,
   so no fold ever influences its own model. Each seed has its own fold split; seeds are averaged.

TabM keeps the earlier project's validated recipe (40 epochs), so it skips step 1. Training it on CPU takes
about an hour per seed, so this base version uses one seed; it serves as the reference for notebook 04,
where the TabM that enters the final blend gets three seeds.
Runs are cached in `work/preds/`; re-running the notebook reuses finished results.

In [1]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
from src.utils import load_features, auc, OUTPUTS
from src.cv import run_cv, median_iterations
from src import models

train, test, sets = load_features()
y = train.TARGET.to_numpy().astype(int)
BASE, RECON = sets['base'], sets['base'] + sets['recon']
SEEDS = (0, 1, 2)
results = {}

LGB_PARAMS = dict(num_leaves=21, min_child_samples=89, colsample_bytree=0.685, subsample=0.962, subsample_freq=1,
                  reg_lambda=2.13, reg_alpha=3.69)            # tuned with Optuna in round 1
CAT_PARAMS = dict(depth=5, l2_leaf_reg=8, random_strength=0.5)  # selected in rounds 2-3

## 1. How much does feature engineering add? (LightGBM, development runs)

In [2]:
_, _, info = run_cv('lgb_raw_dev', models.lightgbm(LGB_PARAMS), train[sets['raw']], y, test[sets['raw']],
                    config=dict(params=LGB_PARAMS))
results['LightGBM, raw 32 columns (dev)'] = info['auc']
_, _, lgb_dev = run_cv('lgb_base_dev', models.lightgbm(LGB_PARAMS), train[BASE], y, test[BASE],
                       config=dict(params=LGB_PARAMS))
results['LightGBM, base 55 columns (dev)'] = lgb_dev['auc']

lgb_raw_dev: OOF AUC 0.75065  per seed [0.75065]  (0.3 min of fitting)


  lgb_base_dev seed 0 fold 0: AUC 0.76058  {'iterations': 609}  (6s)


  lgb_base_dev seed 0 fold 1: AUC 0.75269  {'iterations': 590}  (6s)


  lgb_base_dev seed 0 fold 2: AUC 0.76145  {'iterations': 451}  (5s)


  lgb_base_dev seed 0 fold 3: AUC 0.76651  {'iterations': 533}  (5s)


  lgb_base_dev seed 0 fold 4: AUC 0.76659  {'iterations': 458}  (5s)


lgb_base_dev: OOF AUC 0.76150  per seed [0.7615]  (0.5 min of fitting)


## 2. Final LightGBM (fixed trees, 3 seeds)

In [3]:
n = median_iterations(lgb_dev)
oof_lgb, _, info = run_cv('lgb_base', models.lightgbm(LGB_PARAMS, iterations=n), train[BASE], y, test[BASE],
                          seeds=SEEDS, config=dict(params=LGB_PARAMS, iterations=n))
results['LightGBM base (final)'] = info['auc']

lgb_base: OOF AUC 0.76330  per seed [0.76132, 0.7615, 0.76217]  (1.3 min of fitting)


## 3. CatBoost and CatBoost-recon

In [4]:
_, _, cat_dev = run_cv('cat_base_dev', models.catboost(CAT_PARAMS), train[BASE], y, test[BASE], config=dict(params=CAT_PARAMS))
n = median_iterations(cat_dev)
oof_cat, _, info = run_cv('cat_base', models.catboost(CAT_PARAMS, iterations=n), train[BASE], y, test[BASE],
                          seeds=SEEDS, config=dict(params=CAT_PARAMS, iterations=n))
results['CatBoost base (dev)'], results['CatBoost base (final)'] = cat_dev['auc'], info['auc']

cat_base_dev: OOF AUC 0.76337  per seed [0.76337]  (10.4 min of fitting)
cat_base: OOF AUC 0.76484  per seed [0.76308, 0.76359, 0.76332]  (19.0 min of fitting)


In [5]:
_, _, recon_dev = run_cv('cat_recon_dev', models.catboost(CAT_PARAMS), train[RECON], y, test[RECON], config=dict(params=CAT_PARAMS))
n = median_iterations(recon_dev)
oof_recon, _, info = run_cv('cat_recon', models.catboost(CAT_PARAMS, iterations=n), train[RECON], y, test[RECON],
                            seeds=SEEDS, config=dict(params=CAT_PARAMS, iterations=n))
results['CatBoost-recon (dev)'], results['CatBoost-recon (final)'] = recon_dev['auc'], info['auc']

cat_recon_dev: OOF AUC 0.76329  per seed [0.76329]  (8.0 min of fitting)


cat_recon: OOF AUC 0.76504  per seed [0.76303, 0.76393, 0.76368]  (33.7 min of fitting)


## 4. TabM (neural network)
TabM trains *k = 16* small MLPs that share most weights and averages them (an efficient built-in ensemble).
Numbers pass through piecewise-linear embeddings (each value is encoded by which of 48 quantile bins it
falls in), which lets an MLP learn step-like effects that trees find easily. Inputs: median-filled numbers
+ missing flags, normal-quantile scaled; categories as learned embeddings. AdamW, batch 1024, cosine schedule.

In [6]:
oof_tabm, _, info = run_cv('tabm_base', models.tabm(iterations=40), train[BASE], y, test[BASE], seeds=(0,),
                           config=dict(epochs=40))
results['TabM base (final, 1 seed)'] = info['auc']

tabm_base: OOF AUC 0.76127  per seed [0.76127]  (178.1 min of fitting)


## 5. Summary
Development scores use early stopping on the scored fold (optimistic); final scores do not and average 3 seeds
(TabM: 1 seed).

In [7]:
oofs = dict(LightGBM=oof_lgb, CatBoost=oof_cat, CatBoost_recon=oof_recon, TabM=oof_tabm)
print(pd.Series(results).round(5).to_string())
print('\nCorrelation of OOF predictions (lower = more complementary):')
pd.DataFrame(oofs).corr(method='spearman').round(3)

LightGBM, raw 32 columns (dev)     0.75065
LightGBM, base 55 columns (dev)    0.76150
LightGBM base (final)              0.76330
CatBoost base (dev)                0.76337
CatBoost base (final)              0.76484
CatBoost-recon (dev)               0.76329
CatBoost-recon (final)             0.76504
TabM base (final, 1 seed)          0.76127

Correlation of OOF predictions (lower = more complementary):


,LightGBM,CatBoost,CatBoost_recon,TabM
LightGBM,1.000,0.976,0.971,0.946
CatBoost,0.976,1.000,0.991,0.942
CatBoost_recon,0.971,0.991,1.000,0.938
TabM,0.946,0.942,0.938,1.000
